In [1]:
import json
import numpy as np
from z3 import *
import galois

# Load code

In [2]:
dict_load = json.load(open('../../Unfolded code search/UnfoldedCode.json', 'r'))

# stabilizers
stabilizers = dict_load['stabilzers']

# X_L rotations
rotations = dict_load['rotations']

# information qubits where X physical = X logical
info_qubits = dict_load['info_qubits']

# out qubits hosting magic state
out_qubits = dict_load['out_qubits']

# Add qubits

In [3]:
# add extra qubit and stabilizer for the CNOT ordering
stabilizers.append([19,20])

stabilizers

[[0, 2],
 [1, 0, 3, 4],
 [5, 1],
 [2, 6],
 [3, 2, 7, 8],
 [4, 3, 8, 9],
 [10, 4, 5, 9],
 [6, 7, 11, 12],
 [7, 8, 12, 13],
 [8, 9, 13, 14],
 [15, 9, 10, 14],
 [11, 16],
 [16, 12, 13, 17],
 [17, 13, 14, 18],
 [19, 14, 15, 18],
 [19, 20]]

# Parity check matrix

In [4]:
# convert stabilizers to empty numpy array
max_len = max(map(len, stabilizers))
stabilizers = np.array([sublist + [np.nan] * (max_len - len(sublist)) for sublist in stabilizers])

stabilizers

array([[ 0.,  2., nan, nan],
       [ 1.,  0.,  3.,  4.],
       [ 5.,  1., nan, nan],
       [ 2.,  6., nan, nan],
       [ 3.,  2.,  7.,  8.],
       [ 4.,  3.,  8.,  9.],
       [10.,  4.,  5.,  9.],
       [ 6.,  7., 11., 12.],
       [ 7.,  8., 12., 13.],
       [ 8.,  9., 13., 14.],
       [15.,  9., 10., 14.],
       [11., 16., nan, nan],
       [16., 12., 13., 17.],
       [17., 13., 14., 18.],
       [19., 14., 15., 18.],
       [19., 20., nan, nan]])

In [5]:
# parity check matrix
H = np.zeros((len(stabilizers),int(np.nanmax(stabilizers))+1),dtype = int)
for i, row in enumerate(stabilizers):
    H[i, row[~np.isnan(row)].astype(int)] = 1
        
H

array([[1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
       [1, 1, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
       [0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 1, 1, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 1, 1, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 1, 1, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 1, 1, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 1, 1,

# CNOT order

## SAT solver

In [6]:
# SAT Variables: TimeSteps[i,j,k]=1 -> CNOT of stabilizer j on qubit k at timestep i 
TimeSteps = np.empty((4,len(stabilizers),int(np.nanmax(stabilizers))+1),dtype = object)

for i, j, k in np.ndindex(TimeSteps.shape):
    TimeSteps[i, j, k] = Bool(f"b_{i}_{j}_{k}")

In [7]:
s = SolverFor("QF_FD")

### SAT constraints

# Timesteps sum to H
for i in range(len(H)):
    for j in range(len(H[0])):
        constraint = AtLeast(*TimeSteps[:,i,j], int(H[i][j]))
        s.add(constraint)
        constraint = AtMost(*TimeSteps[:,i,j], int(H[i][j]))
        s.add(constraint)
        
# only one qubit acted on in one timestep
for i in range(TimeSteps.shape[0]):
    for j in range(len(H)):
        constraint = AtMost(*TimeSteps[i,j,:], 1)
        s.add(constraint)
        
# only one cnot per stabilizer in one timestep  
for i in range(TimeSteps.shape[0]):
    for j in range(len(H[0])):
        constraint = AtMost(*TimeSteps[i,:,j], 1)
        s.add(constraint)
        
# weight-2 stab in 2 timesteps               
for i in range(len(H)):
    if np.sum(H[i]) == 2:
        for index1 in range(TimeSteps.shape[0]):
            for index2 in range(TimeSteps.shape[0]):
                if abs(index1 - index2) > 1:
                    constraint = And(AtLeast(*TimeSteps[index1,i,:],1),AtLeast(*TimeSteps[index2,i,:],1))
                    s.add(Not(constraint))

In [8]:
# CNOT gate just before X^1/4 gate
for rotation in rotations:
    LastRound = BoolVal(False)
    for i in range(len(H)):
        LastRound = Or(LastRound,TimeSteps[TimeSteps.shape[0]-1,i,rotation])
    
    s.add(LastRound)

In [9]:
# solve
if s.check() == sat:
    print('sat')
    m = s.model()
else:
    print("unsat")

sat


## retriev result

In [10]:
TimeSteps_result = np.zeros(TimeSteps.shape, dtype=int)
    
for i, j, k in np.ndindex(TimeSteps.shape):
    TimeSteps_result[i, j, k] = is_true(m[TimeSteps[i, j, k]])

In [11]:
stabilizers = np.full((len(stabilizers),4),np.nan)

for i in range(len(stabilizers)):
    for timestep, H in enumerate(TimeSteps_result):
        qubit = np.where(H[i] == 1)[0]
        if len(qubit) == 1:
            stabilizers[i, timestep] = qubit[0]
            
stabilizers

array([[nan, nan,  2.,  0.],
       [ 3.,  1.,  0.,  4.],
       [nan, nan,  5.,  1.],
       [nan, nan,  6.,  2.],
       [ 8.,  2.,  7.,  3.],
       [ 4.,  9.,  3.,  8.],
       [ 5., 10.,  4.,  9.],
       [12.,  6., 11.,  7.],
       [ 7.,  8., 13., 12.],
       [ 9., 13.,  8., 14.],
       [14., 15.,  9., 10.],
       [nan, nan, 16., 11.],
       [13., 16., 12., 17.],
       [18., 14., 17., 13.],
       [19., 18., 14., 15.],
       [nan, nan, 20., 19.]])

# Add output repetition code

In [12]:
stabilizers = stabilizers.tolist()

# add distance d repetition code starting from qubit 18
d = 9
stabilizers.append([18,21,np.nan,np.nan])
out_qubits.append(21)
for i in range(d - 2):
    stabilizers.append([21+i,21+i+1,np.nan,np.nan])
    out_qubits.append(21+i+1)

In [13]:
stabilizers = np.array(stabilizers)
stabilizers

array([[nan, nan,  2.,  0.],
       [ 3.,  1.,  0.,  4.],
       [nan, nan,  5.,  1.],
       [nan, nan,  6.,  2.],
       [ 8.,  2.,  7.,  3.],
       [ 4.,  9.,  3.,  8.],
       [ 5., 10.,  4.,  9.],
       [12.,  6., 11.,  7.],
       [ 7.,  8., 13., 12.],
       [ 9., 13.,  8., 14.],
       [14., 15.,  9., 10.],
       [nan, nan, 16., 11.],
       [13., 16., 12., 17.],
       [18., 14., 17., 13.],
       [19., 18., 14., 15.],
       [nan, nan, 20., 19.],
       [18., 21., nan, nan],
       [21., 22., nan, nan],
       [22., 23., nan, nan],
       [23., 24., nan, nan],
       [24., 25., nan, nan],
       [25., 26., nan, nan],
       [26., 27., nan, nan],
       [27., 28., nan, nan]])

# Calculate logical qubits

In [14]:
# parity check matrix
H = np.zeros((len(stabilizers),int(np.nanmax(stabilizers))+1),dtype = int)
for i, row in enumerate(stabilizers):
    H[i, row[~np.isnan(row)].astype(int)] = 1

# generative matrix
GF2 = galois.GF2
H = GF2(H)
G = H.null_space()
G = np.array(G)

np.set_printoptions(linewidth=np.inf)
G

array([[1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 0, 1, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0],
       [0, 1, 0, 0, 1, 1, 0, 0, 0, 1, 1, 0, 0, 0, 1, 1, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
       [0, 0, 0, 1, 1, 0, 0, 0, 1, 1, 0, 0, 0, 1, 1, 0, 0, 1, 1, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1],
       [0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 0, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]], dtype=uint8)

In [15]:
Logical_qubits = np.zeros(G.shape,dtype = int)

# rearrange order to match sat solver solution
Logical_qubits[0] = (G[0]+G[1]+G[2]+G[3])%2
Logical_qubits[1] = (G[0]+G[1]+G[3]+G[4])%2
Logical_qubits[2] = (G[0]+G[1]+G[2]+G[4])%2
Logical_qubits[3] = (G[1]+G[2])%2
Logical_qubits[4] = (G[1]+G[3]+G[4])%2

Logical_qubits

array([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
       [1, 1, 1, 0, 0, 1, 1, 1, 0, 0, 1, 1, 1, 0, 0, 1, 1, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0],
       [1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
       [0, 1, 0, 1, 0, 1, 0, 0, 1, 0, 1, 0, 0, 1, 0, 1, 0, 1, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0],
       [0, 1, 0, 0, 1, 1, 0, 1, 1, 0, 0, 1, 0, 0, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]])

In [16]:
Logical_qubits = [np.where(Logical_qubits[i])[0].tolist() for i in range(Logical_qubits.shape[0])]

for row in Logical_qubits:
    print(row)

[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
[0, 1, 2, 5, 6, 7, 10, 11, 12, 15, 16, 19, 20]
[0, 1, 2, 3, 4, 5, 6, 11, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28]
[1, 3, 5, 8, 10, 13, 15, 17, 19, 20]
[1, 4, 5, 7, 8, 11, 14, 15, 16, 17]


# Save unfolded code

In [17]:
dict_save = {}
dict_save['stabilzers'] = stabilizers.tolist()
dict_save['rotations'] = rotations
dict_save['info_qubits'] = info_qubits
dict_save['out_qubits'] = out_qubits
dict_save['logical_qubits'] = Logical_qubits

with open('UnfoldedCode_d=9.json', 'w') as json_file:
    json.dump(dict_save, json_file)